In [1]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime
import os
import time
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Retrieve environment variable
base_url = os.getenv("BASE_URL")

# Define bounding box for the region
latitude_center, longitude_center = 46.1512, 14.9955
lat_min, lat_max = latitude_center - 1, latitude_center + 1
lon_min, lon_max = longitude_center - 2, longitude_center + 1.80

# Generate latitude and longitude values
lat_values = np.linspace(lat_min, lat_max, int((lat_max - lat_min) / 0.018))
lon_values = np.linspace(lon_min, lon_max, int((lon_max - lon_min) / 0.018))

# Specify date range
start_date = "2023-08-20"
end_date = "2023-08-20"

# List of daily variables to download
daily_variables = ["temperature_2m_max", "temperature_2m_min", "windgusts_10m_max", "precipitation_sum"]

# Initialize data list
data = []

# Initialize start time
start_time = time.time()

# Display starting message
print("Starting to fetch data...")

# Iterate over latitude and longitude values
for i, lat in enumerate(lat_values):
    for j, lon in enumerate(lon_values):
        # Construct API URL
        url = f"https://{base_url}/v1/dwd-icon?latitude={lat}&longitude={lon}&daily={','.join(daily_variables)}&timezone=Europe%2FLondon&start_date={start_date}&end_date={end_date}"
        
        # Make API request and measure call duration
        call_start_time = time.time()
        response = requests.get(url)
        call_duration = time.time() - call_start_time

        # Retrieve daily data
        daily_data = response.json()['daily']

        # Prepare data row
        row = [lat, lon]
        for var in daily_variables:
            value = max(daily_data.get(var, [])) if var in ["windgusts_10m_max", "precipitation_sum", "temperature_2m_max", "temperature_2m_min"] else sum(daily_data.get(var, []))
            row.append(value)

        data.append(row)

        # Estimate remaining time and display progress
        total_calls = len(lat_values) * len(lon_values)
        approx_total_time = call_duration * total_calls
        approx_remaining_time = approx_total_time - (time.time() - start_time)

        progress_percentage = (i * len(lon_values) + j + 1) / total_calls * 100
        print(f"Approximate remaining time: {approx_remaining_time / 60:.2f} minutes | Progress: {progress_percentage:.2f}% complete")

# Display completion message
print("Data fetching complete.")

# Create DataFrame and save to CSV
columns = ["Latitude", "Longitude"] + daily_variables
df = pd.DataFrame(data, columns=columns)
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
filename = f'all_variables_data_{start_date}_to_{end_date}_{timestamp}.csv'
df.to_csv(filename, index=False)
print(f"Data saved to '{filename}'")


Starting to fetch data...
Approximate remaining time: 369.73 minutes | Progress: 0.00% complete
Approximate remaining time: 208.62 minutes | Progress: 0.01% complete
Approximate remaining time: 391.75 minutes | Progress: 0.01% complete
Approximate remaining time: 205.73 minutes | Progress: 0.02% complete
Approximate remaining time: 208.33 minutes | Progress: 0.02% complete
Approximate remaining time: 376.03 minutes | Progress: 0.03% complete
Approximate remaining time: 209.75 minutes | Progress: 0.03% complete
Approximate remaining time: 204.04 minutes | Progress: 0.03% complete
Approximate remaining time: 363.97 minutes | Progress: 0.04% complete
Approximate remaining time: 218.19 minutes | Progress: 0.04% complete
Approximate remaining time: 203.25 minutes | Progress: 0.05% complete
Approximate remaining time: 361.76 minutes | Progress: 0.05% complete
Approximate remaining time: 216.96 minutes | Progress: 0.06% complete
Approximate remaining time: 221.28 minutes | Progress: 0.06% com